In [ ]:
import os
import re
import sqlite3
import anthropic
import pandas as pd
from dotenv import load_dotenv

# Chave de API (lida do arquivo .env, na mesma pasta do notebook)
load_dotenv(override=True)
assert os.getenv("ANTHROPIC_API_KEY"), "Chave não encontrada: confira o arquivo .env"
client = anthropic.Anthropic()

# Banco: cria o vendas.db a partir do CSV (só na primeira vez)
if not os.path.exists("vendas.db"):
    df = pd.read_csv("vendas_ficticias.csv")
    c = sqlite3.connect("vendas.db")
    df.to_sql("vendas", c, if_exists="replace", index=False)
    c.close()

# Conexão somente leitura: o agente não consegue alterar nem apagar dados
conn = sqlite3.connect("file:vendas.db?mode=ro", uri=True)

# Descrição da tabela que o modelo enxerga
SCHEMA = """Tabela vendas(
    id_venda INT, data TEXT (formato YYYY-MM-DD), uf TEXT, cidade TEXT,
    categoria TEXT, produto TEXT, quantidade INT, preco_unitario REAL,
    valor_total REAL, vendedor TEXT, canal TEXT)"""

SYSTEM = (
    "Você converte perguntas em uma única query SQLite, somente SELECT. "
    "Responda apenas com a query, sem explicação e sem crases. "
    f"Schema: {SCHEMA}"
)


def gerar_sql(pergunta):
    resp = client.messages.create(
        model="claude-sonnet-5-5",
        max_tokens=2000,
        system=SYSTEM,
        messages=[{"role": "user", "content": pergunta}],
    )
    if resp.stop_reason == "max_tokens":
        raise RuntimeError("Resposta cortada: aumente o max_tokens.")
    texto = next(b.text for b in resp.content if b.type == "text")
    return re.sub(r"```(?:sql)?", "", texto).strip()


def perguntar(pergunta):
    sql = gerar_sql(pergunta)
    print("SQL gerado:\n", sql, "\n")

    # só deixa passar SELECT (ou WITH ... SELECT)
    if not sql.lower().startswith(("select", "with")):
        print("Query bloqueada: o agente só pode executar consultas SELECT.")
        return
    try:
        resultado = pd.read_sql_query(sql, conn)
    except Exception as e:
        print("Erro ao executar a query:", e)
        return
    print(resultado)


# Loop de perguntas (deixe vazio ou digite 'sair' para encerrar)
while True:
    pergunta = input("Pergunta: ").strip()
    if pergunta.lower() in ("", "sair"):
        break
    perguntar(pergunta)

conn.close()

SQL gerado:
 SELECT produto, SUM(quantidade) AS total_vendido FROM vendas GROUP BY produto ORDER BY total_vendido DESC LIMIT 10; 

                 produto  total_vendido
0        Bola de Futebol            235
1            Agenda 2026            234
2         Tapete de Yoga            222
3              Air Fryer            187
4         Aspirador Robô            183
5        Jogo de Panelas            181
6  Caderno Universitário            178
7        Mochila Escolar            174
8            Kit Canetas            169
9         Fone Bluetooth            167
SQL gerado:
 SELECT uf, SUM(quantidade) AS total_produtos_vendidos FROM vendas GROUP BY uf ORDER BY total_produtos_vendidos DESC; 

   uf  total_produtos_vendidos
0  SP                     1149
1  RJ                      441
2  MG                      417
3  PR                      304
4  RS                      286
5  BA                      285
SQL gerado:
 SELECT vendedor, SUM(valor_total) AS total_vendido FROM vendas GROU